# WDR91 DEL Screen — Exploratory Data Analysis

**Goal:** Understand the structure of the DNA-Encoded Library (DEL) screen data for the WDR91 target protein.

**Dataset:** `WDR91.parquet` — 375,595 DEL compounds pre-screened against WDR91, with binary hit labels and pre-computed fingerprints.

---
**Before running:** Upload `WDR91.parquet` to your Google Drive under `My Drive/CS502/data/WDR91.parquet`

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install pyarrow tqdm seaborn -q

In [ ]:
# ── 2. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/CS502/data/WDR91.parquet'

In [ ]:
# ── 3. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

## 1. Schema Inspection
Read just the schema — no data loaded yet.

In [ ]:
pf = pq.ParquetFile(DATA_PATH)
print('Schema:\n', pf.schema_arrow)
print(f'\nTotal rows : {pf.metadata.num_rows:,}')
print(f'Row groups : {pf.metadata.num_row_groups}')

## 2. Load Scalar Columns
Skip the fingerprint arrays (stored as lists) to keep memory low.

In [ ]:
SCALAR_COLS = [
    'COMPOUND_ID', 'LIBRARY_ID', 'BB1_ID', 'BB2_ID', 'BB3_ID',
    'TARGET_ID', 'TARGET_VALUE', 'LABEL', 'MW', 'ALOGP'
]

df = pq.read_table(DATA_PATH, columns=SCALAR_COLS).to_pandas()
df['lib_prefix'] = df['LIBRARY_ID'].str[:3]
df['log1p_target'] = np.log1p(df['TARGET_VALUE'])

print(f'Shape: {df.shape}')
df.head()

## 3. Class Balance

In [ ]:
label_counts = df['LABEL'].value_counts()
print(label_counts)
print(f'\nHit rate: {df["LABEL"].mean()*100:.2f}%  ({label_counts[1]:,} hits / {len(df):,} total)')
print(f'Imbalance ratio: {label_counts[0]/label_counts[1]:.1f}:1 (non-hit:hit)')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Non-hit (0)', 'Hit (1)'], label_counts.values, color=['steelblue', 'coral'])
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
for i, v in enumerate(label_counts.values):
    ax.text(i, v * 1.01, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 4. TARGET_VALUE Distribution
The raw DEL read count — how many DNA barcodes were sequenced for each compound.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# All compounds
axes[0].hist(df['TARGET_VALUE'], bins=50, color='steelblue', log=True)
axes[0].set_title('TARGET_VALUE (all, log y-scale)')
axes[0].set_xlabel('Read count')

# Hits only
hits = df[df['LABEL'] == 1]
axes[1].hist(hits['TARGET_VALUE'], bins=50, color='coral')
axes[1].set_title(f'TARGET_VALUE — Hits only (n={len(hits):,})')
axes[1].set_xlabel('Read count')

# Log1p transform
axes[2].hist(hits['log1p_target'], bins=50, color='seagreen')
axes[2].set_title('log1p(TARGET_VALUE) — Hits only')
axes[2].set_xlabel('log1p(count)')

plt.tight_layout()
plt.show()

print('Hit TARGET_VALUE stats:')
print(hits['TARGET_VALUE'].describe())

## 5. DEL Library Breakdown
The dataset contains compounds from 39 different DEL sub-libraries.

In [ ]:
lib_stats = df.groupby('lib_prefix').agg(
    total=('LABEL', 'count'),
    hits=('LABEL', 'sum')
).assign(hit_rate=lambda x: x.hits / x.total * 100).sort_values('total', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(lib_stats.index, lib_stats['total'], color='steelblue')
axes[0].set_xlabel('Compound count')
axes[0].set_title('Compounds per DEL Library')
axes[0].invert_yaxis()

axes[1].barh(lib_stats.index, lib_stats['hit_rate'], color='coral')
axes[1].set_xlabel('Hit rate (%)')
axes[1].set_title('Hit Rate per DEL Library')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(lib_stats.head(10).to_string())

## 6. Building Block Analysis

DEL compounds are made from 3 building blocks (BB1, BB2, BB3). Some BBs may be systematically enriched regardless of the full molecule — a key signal for modeling.

In [ ]:
print('Unique building blocks:')
for bb in ['BB1_ID', 'BB2_ID', 'BB3_ID']:
    print(f'  {bb}: {df[bb].nunique():,}')

# Top enriched BBs for each position
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, bb in zip(axes, ['BB1_ID', 'BB2_ID', 'BB3_ID']):
    bb_stats = df.groupby(bb)['LABEL'].agg(['sum', 'count'])
    bb_stats['hit_rate'] = bb_stats['sum'] / bb_stats['count'] * 100
    bb_stats = bb_stats[bb_stats['count'] >= 50]  # min 50 occurrences
    top20 = bb_stats.nlargest(20, 'hit_rate')
    ax.barh(top20.index, top20['hit_rate'], color='mediumpurple')
    ax.set_title(f'Top 20 {bb} by Hit Rate')
    ax.set_xlabel('Hit rate (%)')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Physicochemical Properties (MW & ALogP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in zip(axes, ['MW', 'ALOGP'], ['Molecular Weight', 'ALogP (lipophilicity)']):
    ax.hist(df[df['LABEL']==0][col], bins=60, alpha=0.6, label='Non-hit', color='steelblue', density=True)
    ax.hist(df[df['LABEL']==1][col], bins=60, alpha=0.7, label='Hit', color='coral', density=True)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()
plt.tight_layout()
plt.show()

print('MW stats by label:')
print(df.groupby('LABEL')['MW'].describe())
print('\nALogP stats by label:')
print(df.groupby('LABEL')['ALOGP'].describe())

## 8. Summary

| Property | Value |
|---|---|
| Total compounds | 375,595 |
| Hits (LABEL=1) | 28,778 (7.66%) |
| Imbalance ratio | 12.1 : 1 |
| DEL sub-libraries | 39 |
| Unique BB1 / BB2 / BB3 | 2,485 / 2,319 / 3,562 |
| Fingerprints available | ECFP4/6, FCFP4/6, MACCS, RDK, AVALON, ATOMPAIR, TOPTOR |
| MW range | 253 – 786 Da |

**Key observations:**
- Hit labels are perfectly determined by TARGET_VALUE > 0 (clean separation)
- Hit rate varies greatly across DEL libraries — library identity is a strong signal
- Some building blocks are systematically enriched — good feature to add
- MW and ALogP distributions are similar for hits vs non-hits — physicochemistry alone won't discriminate well
- Fingerprints are the main feature for modeling